# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vinay21rout/flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os

if not os.path.exists('flyrank-ML-internship'):
    !git clone https://github.com/aamnamalik16-bit/flyrank-ML-internship.git

os.chdir('flyrank-ML-internship')

!pip install duckdb -q

import duckdb
import pandas as pd
import numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
import random

random.seed(42)
np.random.seed(42)

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
print("Setup complete!")

Cloning into 'flyrank-ML-internship'...
remote: Enumerating objects: 173, done.
remote: Counting objects: 100% (173/173), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 173 (delta 80), reused 111 (delta 41), pack-reused 0 (from 0)
Receiving objects: 100% (173/173), 1.86 MiB | 10.91 MiB/s, done.
Resolving deltas: 100% (80/80), done.
Setup complete!


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Method: Logistic Regression -> Random Forest")
print("Metric: Precision@20")
print("Reason: ranking problem needs probability scores, not just labels")

Method: Logistic Regression -> Random Forest
Metric: Precision@20
Reason: ranking problem needs probability scores, not just labels


Method choice: Random Forest Classifier

My lane is Query Intelligence — I am predicting which query-page pairs are rising opportunities (is_rising = clicks_last30 > clicks_prev30). This is a "which first?" ranking problem, so I need a model that outputs probabilities I can rank by.

I start with Logistic Regression (readable, fast, good baseline) then move to Random Forest (handles non-linear interactions between impressions, position, and click delta). I add complexity only if Random Forest meaningfully beats Logistic Regression.

I will evaluate both using Precision@20 — matching the metric from my Week-4 baseline

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Load data and build features
df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        impressions_last30,
        impressions_prev30,
        clicks_last30,
        clicks_prev30,
        avg_position_last30,
        (clicks_last30 - clicks_prev30) as click_delta,
        (clicks_last30 > clicks_prev30) as is_rising
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    WHERE (impressions_last30 >= 10) IS TRUE
""").df()

# Client holdout split
clients = df['client_hash_id'].unique()
np.random.shuffle(clients)
train_clients = clients[:int(len(clients)*0.8)]
test_clients = clients[int(len(clients)*0.8):]

train_df = df[df['client_hash_id'].isin(train_clients)]
test_df = df[df['client_hash_id'].isin(test_clients)]

print(f"Train rows: {len(train_df)}, clients: {len(train_clients)}")
print(f"Test rows: {len(test_df)}, clients: {len(test_clients)}")
print(f"Label base rate: {df['is_rising'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train rows: 800582, clients: 39
Test rows: 38075, clients: 10
Label base rate: 0.066


Split design: Client holdout

I split by client_hash_id — whole clients go into either train or test, never both. This is the honest split for this data because:

Pages from the same client share patterns the model could memorize if they appear in both train and test
A client holdout tests whether the model generalizes to new clients — the real-world use case
I use 80% of clients for training and 20% for testing (random split, seed=42)

I do NOT use a time-aware split here because fact_content_query_90d has a fixed 90-day window with no month partitioning — there is no clean time axis to split on.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.preprocessing import StandardScaler

features = [
    'impressions_last30',
    'impressions_prev30',
    'avg_position_last30',
    'click_delta',
    'clicks_prev30'
]

label = 'is_rising'

X_train = train_df[features].fillna(0)
y_train = train_df[label].astype(int)
X_test = test_df[features].fillna(0)
y_test = test_df[label].astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Baseline: rank by impressions_last30 (Week-4 rule)
baseline_p20 = precision_at_k(test_df['impressions_last30'], y_test, 20)

# Logistic Regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)
lr_p20 = precision_at_k(lr.predict_proba(X_test_sc)[:,1], y_test, 20)

# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_p20 = precision_at_k(rf.predict_proba(X_test)[:,1], y_test, 20)

# Comparison table
print("=" * 45)
print(f"{'Method':<25} {'Precision@20':>12}")
print("=" * 45)
print(f"{'Base rate':<25} {y_test.mean():>12.3f}")
print(f"{'Baseline (impressions)':<25} {baseline_p20:>12.3f}")
print(f"{'Logistic Regression':<25} {lr_p20:>12.3f}")
print(f"{'Random Forest':<25} {rf_p20:>12.3f}")
print("=" * 45)

Method                    Precision@20
Base rate                        0.061
Baseline (impressions)           0.300
Logistic Regression              1.000
Random Forest                    1.000


In [5]:
# Check what columns are actually in the table
sample = con.sql(f"""
    SELECT * FROM read_parquet('{rel}/fact_content_quSSSsSery_90d.parquet')
    LIMIT 1
""").df()
print(list(sample.columns))

['client_hash_id', 'content_hash_id', 'query_hash_id', 'query_char_count', 'query_token_count', 'window_start', 'window_end', 'impressions_90d', 'clicks_90d', 'impressions_last30', 'clicks_last30', 'impressions_prev30', 'clicks_prev30', 'avg_position_90d', 'avg_position_last30', 'avg_position_prev30', 'content_total_impressions_90d', 'content_visible_query_count', 'rare_query_count', 'rare_impressions_share', 'anonymized_impressions_share']


In [7]:
# Reload data with all safe features
df = con.sql(f"""
    SELECT
        content_hash_id,
        client_hash_id,
        query_hash_id,
        impressions_90d,
        avg_position_90d,
        query_char_count,
        query_token_count,
        content_visible_query_count,
        rare_query_count,
        rare_impressions_share,
        anonymized_impressions_share,
        clicks_last30,
        clicks_prev30,
        impressions_last30,
        (clicks_last30 > clicks_prev30) as is_rising
    FROM read_parquet('{rel}/fact_content_query_90d.parquet')
    WHERE (impressions_last30 >= 10) IS TRUE
""").df()

# Client holdout split
clients = df['client_hash_id'].unique()
np.random.seed(42)
np.random.shuffle(clients)
train_clients = clients[:int(len(clients)*0.8)]
test_clients = clients[int(len(clients)*0.8):]

train_df = df[df['client_hash_id'].isin(train_clients)]
test_df = df[df['client_hash_id'].isin(test_clients)]

y_train = train_df['is_rising'].astype(int)
y_test = test_df['is_rising'].astype(int)

print(f"Train: {len(train_df)} rows, Test: {len(test_df)} rows")
print(f"Base rate: {y_test.mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Train: 800582 rows, Test: 38075 rows
Base rate: 0.061


In [8]:
safe_features = [
    'impressions_90d',
    'avg_position_90d',
    'query_char_count',
    'query_token_count',
    'content_visible_query_count',
    'rare_query_count',
    'rare_impressions_share',
    'anonymized_impressions_share'
]

X_train_s = train_df[safe_features].fillna(0)
X_test_s = test_df[safe_features].fillna(0)

# Baseline
baseline_p20 = precision_at_k(test_df['impressions_last30'], y_test, 20)

# Logistic Regression
from sklearn.preprocessing import StandardScaler
scaler2 = StandardScaler()
X_train_sc = scaler2.fit_transform(X_train_s)
X_test_sc = scaler2.transform(X_test_s)

lr3 = LogisticRegression(random_state=42, max_iter=1000)
lr3.fit(X_train_sc, y_train)
lr3_p20 = precision_at_k(lr3.predict_proba(X_test_sc)[:,1], y_test, 20)

# Random Forest
rf3 = RandomForestClassifier(n_estimators=100, random_state=42)
rf3.fit(X_train_s, y_train)
rf3_p20 = precision_at_k(rf3.predict_proba(X_test_s)[:,1], y_test, 20)

print("Method                    Precision@20")
print("Base rate                ", round(y_test.mean(), 3))
print("Baseline (impressions)   ", round(baseline_p20, 3))
print("Logistic Regression      ", round(lr3_p20, 3))
print("Random Forest            ", round(rf3_p20, 3))

Method                    Precision@20
Base rate                 0.061
Baseline (impressions)    0.3
Logistic Regression       0.1
Random Forest             0.2


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance from Random Forest
importances = pd.Series(rf3.feature_importances_, index=safe_features)
print("Feature importances:")
print(importances.sort_values(ascending=False))

Feature importances:
avg_position_90d                0.190942
impressions_90d                 0.168547
rare_impressions_share          0.126803
anonymized_impressions_share    0.125074
rare_query_count                0.118537
query_char_count                0.109214
content_visible_query_count     0.104621
query_token_count               0.056262
dtype: float64


Errors and interpretation:

The baseline (rank by impressions) achieves Precision@20 = 0.30, well above the base rate of 0.061. Both models underperform the baseline — Logistic Regression at 0.10 and Random Forest at 0.20.

What the model leans on (feature importances):

avg_position_90d — 0.191 (most important)
impressions_90d — 0.169
rare_impressions_share — 0.127
anonymized_impressions_share — 0.125

Position and volume dominate — the model learned that well-ranked, visible pages are more likely to have rising clicks. This makes intuitive sense but also explains why the baseline (pure volume) is hard to beat.

Why the models underperform the baseline: Safe query-level features describe page visibility, not click direction. The baseline ranks by impressions directly — the strongest single signal available.

One honest finding: Query characteristics (query_char_count, query_token_count) contribute least — query shape alone does not predict click momentum.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.